# Notebook 02 — Baseline Models
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polar-bear-after-lunch/AareML/blob/main/notebooks/02_baselines.ipynb)

Establishes three statistical baselines (Persistence, Climatology, Ridge Regression) with block-bootstrap 95% confidence intervals on gauge 2473 test set (2017–2020).

# AareML — 02 Baselines

**Project:** AareML — Predicting River Water Quality in Swiss Catchments  
**Dataset:** CAMELS-CH-Chem (Nascimento et al., 2025)  
**Reference benchmark:** LakeBeD-US (McAfee et al., 2025)

---

Three baselines evaluated on the **dissolved oxygen (DO)** and **water temperature**
forecasting task at gauge 2473 (test set 2017–2020):

| # | Baseline | Description |
|---|----------|-------------|
| 1 | **Persistence** | Last observed value repeated for all 14 forecast steps |
| 2 | **Climatology** | Day-of-year median from training data |
| 3 | **Ridge Regression** | Ridge on flattened 21-day lookback window |

Metrics: RMSE, MAE, NSE, KGE — with 95% block-bootstrap confidence intervals.
Task mirrors the LakeBeD-US benchmark (21-day lookback, 14-day horizon).


## 0 · Setup


In [ ]:
# ── Colab setup (auto-runs only in Google Colab) ──────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import os
    from pathlib import Path

    # ── 1. Clone repo ──────────────────────────────────────────────────────
    if not Path('AareML').exists():
        os.system('git clone https://github.com/polar-bear-after-lunch/AareML.git')
    if not str(Path.cwd()).endswith('AareML'):
        os.chdir('AareML')

    # ── 2. Install dependencies ────────────────────────────────────────────
    os.system('pip install -q -r requirements.txt')

    # ── 3. Mount Google Drive for persistent data storage ─────────────────
    from google.colab import drive
    drive.mount('/content/drive')

    # Data stored in Drive so it survives across sessions (~360 MB total)
    DRIVE_DATA = Path('/content/drive/MyDrive/AareML_data')
    LOCAL_DATA = Path('data')
    LOCAL_DATA.mkdir(exist_ok=True)

    # ── 4. CAMELS-CH-Chem ─────────────────────────────────────────────────
    DRIVE_CAMELS = DRIVE_DATA / 'camels-ch-chem'
    LOCAL_CAMELS = LOCAL_DATA / 'camels-ch-chem'

    if DRIVE_CAMELS.exists():
        # Already in Drive — symlink to local path (fast, no re-download)
        if not LOCAL_CAMELS.exists():
            os.system(f'ln -s {DRIVE_CAMELS} {LOCAL_CAMELS}')
        print('CAMELS-CH-Chem loaded from Google Drive.')
    else:
        # First time — download to Drive
        print('Downloading CAMELS-CH-Chem to Google Drive (~165 MB, one-time)...')
        DRIVE_DATA.mkdir(parents=True, exist_ok=True)
        os.system(
            'wget -q --show-progress -O /tmp/camels.zip '
            '"https://zenodo.org/api/records/14980027/files/camels-ch-chem.zip/content"'
        )
        os.system(f'unzip -q /tmp/camels.zip -d {DRIVE_CAMELS}')
        os.system('rm /tmp/camels.zip')
        os.system(f'ln -s {DRIVE_CAMELS} {LOCAL_CAMELS}')
        print('CAMELS-CH-Chem saved to Google Drive for future sessions.')

    print(f'Setup complete. Working directory: {os.getcwd()}')


In [ ]:
# ── CPU thread optimisation ────────────────────────────────────────────────
# PyTorch LSTM on CPU is fastest with 4-6 threads — beyond that,
# thread synchronisation overhead outweighs the parallelism gains.
import os, multiprocessing
N_CPU = multiprocessing.cpu_count()
N_THREADS = min(N_CPU, 6)  # clamp to 6 — empirically optimal on macOS
os.environ['OMP_NUM_THREADS']      = str(N_THREADS)
os.environ['MKL_NUM_THREADS']      = str(N_THREADS)
os.environ['OPENBLAS_NUM_THREADS'] = str(N_THREADS)
print(f'CPU cores: {N_CPU} logical | Using {N_THREADS} threads for PyTorch')


In [ ]:
import sys
sys.path.insert(0, '..')   # so notebooks can import src/

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from src.config  import *
from src.data    import load_gauge, load_metadata, preprocess, train_val_test_split, make_windows
from src.metrics import (
    rmse_per_step, mean_rmse, mean_mae, nse, kge,
    block_bootstrap_ci, metrics_table,
)

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 130})


## 1 · Load Focus-Gauge Data


In [ ]:
raw  = load_gauge(FOCUS_GAUGE)
data = preprocess(raw)
train, val, test = train_val_test_split(data)

# Training-set means used for imputation (no leakage)
# Compute training means without duplicated index (FEATURES and TARGETS overlap)
train_means = (
    pd.concat([train[FEATURES].mean(), train[TARGETS].mean()])
    .groupby(level=0).first()   # deduplicate temp_sensor / O2C_sensor
)

print(f'Gauge {FOCUS_GAUGE}')
print(f'Train: {train.index.min().date()} → {train.index.max().date()}  ({len(train):,} days)')
print(f'Val  : {val.index.min().date()}   → {val.index.max().date()}  ({len(val):,} days)')
print(f'Test : {test.index.min().date()}  → {test.index.max().date()}  ({len(test):,} days)')
print()
print('DO non-null (raw):', f"{raw['O2C_sensor'].notna().mean()*100:.1f}%")


## 2 · Gauge Metadata


In [ ]:
meta = load_metadata()
gauge_row = meta[meta['gauge_id'].astype(str) == str(FOCUS_GAUGE)]
if not gauge_row.empty:
    print('Focus gauge metadata:')
    print(gauge_row.T.to_string())
else:
    print(f'Gauge {FOCUS_GAUGE} not found in metadata — check column name')
    print('Metadata columns:', meta.columns.tolist())


## 3 · Build Sliding Windows


In [ ]:
X_train, y_train, d_train = make_windows(train, train_means)
X_val,   y_val,   d_val   = make_windows(val,   train_means)
X_test,  y_test,  d_test  = make_windows(test,  train_means)

print(f'Train: X={X_train.shape}  y={y_train.shape}')
print(f'Val  : X={X_val.shape}    y={y_val.shape}')
print(f'Test : X={X_test.shape}   y={y_test.shape}')


## 4 · Baseline 1 — Persistence

The last observed value of each target in the lookback window is repeated
flat across the entire 14-day forecast horizon.


In [ ]:
tgt_feat_idx = [FEATURES.index(t) for t in TARGETS]
last_val = X_test[:, -1, :][:, tgt_feat_idx]          # [N, n_tgt]
y_pred_persist = np.tile(last_val[:, np.newaxis, :], (1, HORIZON, 1))

print('Persistence — Test set')
for t in TARGETS:
    i = TARGETS.index(t)
    rmse = np.sqrt(((y_test[:,:,i] - y_pred_persist[:,:,i])**2).mean())
    print(f'  {TARGET_LABELS[t]:12s}  RMSE = {rmse:.4f}')


## 5 · Baseline 2 — Climatology

For each calendar day-of-year, the training-set **median** is forecast for
every horizon step that falls on that DOY.  
Vectorised lookup — no Python-level loops over windows.


In [ ]:
# Build DOY → median table from training data
clim = {}
for t in TARGETS:
    s = train[t].dropna()
    clim[t] = (
        s.groupby(s.index.dayofyear).median()
         .reindex(range(1, 367))
         .interpolate('linear').ffill().bfill()
         .values  # length-366 array, index = DOY-1
    )

# Vectorised: compute DOY for each (window_start + horizon_step)
# d_test is DatetimeIndex of length N
n_test = len(d_test)
y_pred_clim = np.empty((n_test, HORIZON, len(TARGETS)), dtype=np.float32)

for h in range(HORIZON):
    forecast_dates = d_test + pd.Timedelta(days=h)
    doys = forecast_dates.dayofyear.values - 1      # 0-based index into clim array
    for j, t in enumerate(TARGETS):
        y_pred_clim[:, h, j] = clim[t][doys]

print('Climatology — Test set')
for t in TARGETS:
    i = TARGETS.index(t)
    rmse = np.sqrt(((y_test[:,:,i] - y_pred_clim[:,:,i])**2).mean())
    print(f'  {TARGET_LABELS[t]:12s}  RMSE = {rmse:.4f}')


## 6 · Baseline 3 — Ridge Regression

Ridge regression trained **per target per horizon step** (14 × 2 = 28 models).
Input: flattened 21-day lookback window (84 features), standardised on training data.
Alpha tuned on the validation set; final model fit on train+val.


In [ ]:
from tqdm.auto import tqdm
X_tr_flat = X_train.reshape(len(X_train), -1)
X_va_flat = X_val.reshape(len(X_val),   -1)
X_te_flat = X_test.reshape(len(X_test),  -1)

ridge_scaler = StandardScaler().fit(X_tr_flat)
X_tr_s = ridge_scaler.transform(X_tr_flat)
X_va_s = ridge_scaler.transform(X_va_flat)
X_te_s = ridge_scaler.transform(X_te_flat)

# Tune alpha on validation set
best_alpha, best_val_rmse = 1.0, np.inf
for alpha in tqdm([0.01, 0.1, 1.0, 10.0, 100.0], desc="Ridge α search"):
    preds_va = np.zeros_like(y_val)
    for h in range(HORIZON):
        for j in range(len(TARGETS)):
            m = Ridge(alpha=alpha).fit(X_tr_s, y_train[:, h, j])
            preds_va[:, h, j] = m.predict(X_va_s)
    score = rmse_per_step(y_val, preds_va).mean()
    if score < best_val_rmse:
        best_val_rmse, best_alpha = score, alpha

print(f'Best alpha = {best_alpha}  (val mean RMSE = {best_val_rmse:.4f})')

# Refit on train+val
X_tv_s = ridge_scaler.transform(np.vstack([X_tr_flat, X_va_flat]))
y_tv   = np.concatenate([y_train, y_val], axis=0)

ridge_models  = [[None]*len(TARGETS) for _ in range(HORIZON)]
y_pred_ridge  = np.zeros_like(y_test)
for h in range(HORIZON):
    for j in range(len(TARGETS)):
        m = Ridge(alpha=best_alpha).fit(X_tv_s, y_tv[:, h, j])
        ridge_models[h][j] = m
        y_pred_ridge[:, h, j] = m.predict(X_te_s)

print('Ridge — Test set')
for t in TARGETS:
    i = TARGETS.index(t)
    rmse = np.sqrt(((y_test[:,:,i] - y_pred_ridge[:,:,i])**2).mean())
    print(f'  {TARGET_LABELS[t]:12s}  RMSE = {rmse:.4f}')


## 7 · Full Metrics Table with Bootstrap CIs

95% confidence intervals via temporal block bootstrap (block=30 days, 500 replicates).


In [ ]:
models_dict = {
    'Persistence':      y_pred_persist,
    'Climatology':      y_pred_clim,
    'Ridge Regression': y_pred_ridge,
}

results_df = metrics_table(models_dict, y_test, n_boot=500)
print(results_df.to_string(index=False))

RESULTS_DIR.mkdir(exist_ok=True)
results_df.to_csv(RESULTS_DIR / 'baseline_results.csv', index=False)
print('\nSaved \u2192 results/baseline_results.csv')

# Save per-gauge Ridge RMSE for use in notebook 04 significance test
# Load each gauge independently (avoids keeping all data in memory)
from src.data import load_gauge, gauges_with_do
ridge_rows = []
all_gauge_ids = gauges_with_do(min_coverage=0.10)
print(f'Computing per-gauge Ridge RMSEs across {len(all_gauge_ids)} gauges...')
for gid in all_gauge_ids:
    try:
        raw_g = load_gauge(gid)
        gdf = preprocess(raw_g)
        _, _, test_g = train_val_test_split(gdf)
        if len(test_g) == 0:
            continue
        X_te = test_g[FEATURES].fillna(train[FEATURES].mean()).values
        y_te = test_g[[t for t in TARGETS if 'do' in t.lower()]].values
        if y_te.shape[1] == 0 or np.all(np.isnan(y_te)):
            continue
        yp = ridge_model.predict(X_te)
        # RMSE on DO target only (first target column)
        valid = ~np.isnan(y_te[:, 0])
        if valid.sum() == 0:
            continue
        rmse = float(np.sqrt(np.mean((yp[valid, 0] - y_te[valid, 0])**2)))
        ridge_rows.append({'gauge_id': str(gid), 'rmse_do': rmse})
    except Exception as e:
        pass  # skip gauges with insufficient data

if ridge_rows:
    pd.DataFrame(ridge_rows).to_csv(RESULTS_DIR / 'baseline_per_gauge.csv', index=False)
    print(f'Saved per-gauge Ridge RMSEs: {len(ridge_rows)} gauges \u2192 results/baseline_per_gauge.csv')
else:
    print('Warning: no per-gauge Ridge results saved')

## 8 · RMSE by Forecast Horizon


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
steps  = np.arange(1, HORIZON + 1)
colors = {'Persistence': '#e07b39', 'Climatology': '#4e9ab5', 'Ridge Regression': '#5d9e5a'}

for ax_i, (t_i, t) in enumerate(enumerate(TARGETS)):
    ax = axes[ax_i]
    for name, y_pred in models_dict.items():
        per = rmse_per_step(y_test, y_pred)[:, t_i]
        ax.plot(steps, per, marker='o', markersize=4,
                label=name, color=colors[name], linewidth=1.8)
    ax.set_xlabel('Forecast horizon (days)', fontsize=11)
    ax.set_ylabel('RMSE', fontsize=11)
    ax.set_title(TARGET_LABELS[t], fontsize=12)
    ax.set_xticks(steps)
    ax.legend(fontsize=9)

fig.suptitle('Baseline RMSE by Forecast Horizon — Gauge 2473 (Test Set)',
             fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '02_baseline_rmse_by_horizon.png', bbox_inches='tight')
plt.show()


## 9 · Example Forecasts


In [ ]:
# Pick winter and summer windows safely (find closest available date)
def find_nearest_idx(dates, target_date):
    diffs = np.abs((dates - pd.Timestamp(target_date)).days)
    return int(np.argmin(diffs))

winter_idx = find_nearest_idx(d_test, '2018-01-15')
summer_idx = find_nearest_idx(d_test, '2018-07-15')
print(f'Winter window: {d_test[winter_idx].date()}')
print(f'Summer window: {d_test[summer_idx].date()}')

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
steps_arr = np.arange(1, HORIZON + 1)

for row, (label, idx) in enumerate([('Winter 2018', winter_idx), ('Summer 2018', summer_idx)]):
    for col, (t_i, t) in enumerate(enumerate(TARGETS)):
        ax = axes[row][col]
        truth = y_test[idx, :, t_i]
        ax.plot(steps_arr, truth, 'k-', linewidth=2, label='Observed')
        for name, y_pred in models_dict.items():
            ax.plot(steps_arr, y_pred[idx, :, t_i], '--',
                    color=colors[name], label=name)
        ax.set_title(f'{label} — {TARGET_LABELS[t]}', fontsize=10)
        ax.set_xlabel('Forecast day', fontsize=9)
        ax.set_ylabel(TARGET_LABELS[t], fontsize=9)
        ax.legend(fontsize=8)

fig.suptitle('Example Baseline Forecasts — Gauge 2473', fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '02_baseline_example_forecasts.png', bbox_inches='tight')
plt.show()


## 10 · Ridge Coefficient Inspection


In [ ]:
feat_names_flat = [f'{f}[t-{LOOKBACK-i}]' for i in range(LOOKBACK) for f in FEATURES]
coefs = ridge_models[0][0].coef_
top_idx   = np.argsort(np.abs(coefs))[-20:][::-1]
top_names = [feat_names_flat[i] for i in top_idx]
top_coefs = coefs[top_idx]

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['#e07b39' if c > 0 else '#4e9ab5' for c in top_coefs]
ax.barh(top_names[::-1], top_coefs[::-1], color=bar_colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Ridge coefficient', fontsize=11)
ax.set_title('Top-20 Features — 1-day-ahead DO forecast', fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '02_ridge_coefficients.png', bbox_inches='tight')
plt.show()


## 11 · Summary


In [ ]:
print('=' * 65)
print('BASELINE SUMMARY — Gauge 2473 (Test Set 2017–2020)')
print('=' * 65)
print(results_df.to_string(index=False))
print('=' * 65)
print('LakeBeD-US seq2seq LSTM reference: DO RMSE = 1.40 mg/L')
